In [2]:
import pandas as pd
import numpy as np
import json
import pickle
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sentence_transformers import SentenceTransformer

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# Load credit card data
df_fraud = pd.read_csv("../data/creditcard.csv")

# Prepare features and target
X_fraud = df_fraud.drop("Class", axis=1).copy()
y_fraud = df_fraud["Class"]

# Scale Amount and Time only
scaler_fraud = StandardScaler()
X_fraud[["Amount", "Time"]] = scaler_fraud.fit_transform(
    X_fraud[["Amount", "Time"]])

# Train test split with stratify
X_train, X_test, y_train, y_test = train_test_split(
    X_fraud, y_fraud,
    test_size=0.2,
    random_state=42,
    stratify=y_fraud
)

# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Train final fraud model
fraud_model = LogisticRegression(max_iter=1000, random_state=42)
fraud_model.fit(X_train_smote, y_train_smote)

# Save fraud model and its scaler to disk
# pickle converts the trained Python object into a binary file
with open("../api/models/fraud_model.pkl", "wb") as f:
    pickle.dump(fraud_model, f)

with open("../api/models/scaler_fraud.pkl", "wb") as f:
    pickle.dump(scaler_fraud, f)

print("Fraud model saved successfully.")
print(f"Model trained on {X_train_smote.shape[0]} samples after SMOTE.")

Fraud model saved successfully.
Model trained on 454902 samples after SMOTE.


In [4]:
# Load superstore data
df_store = pd.read_csv("../data/Sample - Superstore.csv",
                       parse_dates=["Order Date", "Ship Date"],
                       encoding="latin-1")

# Features for anomaly detection
features = ["Sales", "Quantity", "Discount", "Profit"]
X_anomaly = df_store[features].copy()

# Scale all four features
scaler_anomaly = StandardScaler()
X_anomaly_scaled = scaler_anomaly.fit_transform(X_anomaly)

# Train Isolation Forest
anomaly_model = IsolationForest(contamination=0.01,
                                random_state=42,
                                n_estimators=100)
anomaly_model.fit(X_anomaly_scaled)

# Save anomaly model and its scaler
with open("../api/models/anomaly_model.pkl", "wb") as f:
    pickle.dump(anomaly_model, f)

with open("../api/models/scaler_anomaly.pkl", "wb") as f:
    pickle.dump(scaler_anomaly, f)

print("Anomaly model saved successfully.")

Anomaly model saved successfully.


In [5]:
# Rebuild superstore metrics for knowledge base
df_store["Profit_Margin"] = (
    df_store["Profit"] / df_store["Sales"] * 100).round(2)

# Anomaly labels for knowledge base context
X_anomaly_scaled_kb = scaler_anomaly.transform(X_anomaly)
df_store["Anomaly"] = anomaly_model.predict(X_anomaly_scaled_kb)
df_store["Anomaly_Score"] = anomaly_model.decision_function(X_anomaly_scaled_kb)
df_store["Anomaly_Type"] = "Normal"
df_store.loc[(df_store["Anomaly"] == -1) &
             (df_store["Profit"] > 0), "Anomaly_Type"] = "Opportunity"
df_store.loc[(df_store["Anomaly"] == -1) &
             (df_store["Profit"] <= 0), "Anomaly_Type"] = "Risk"

# Regional and category metrics
region_metrics = df_store.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2)
region_metrics["Profit_Margin"] = (
    region_metrics["Total_Profit"] /
    region_metrics["Total_Sales"] * 100).round(2)

category_metrics = df_store.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2)
category_metrics["Profit_Margin"] = (
    category_metrics["Total_Profit"] /
    category_metrics["Total_Sales"] * 100).round(2)

risk_counts = df_store[df_store["Anomaly_Type"] == "Risk"].groupby(
    "Region").size()
opportunity_counts = df_store[df_store["Anomaly_Type"] == "Opportunity"].groupby(
    "Region").size()

# Reference points
best_region = region_metrics["Profit_Margin"].idxmax()
worst_region = region_metrics["Profit_Margin"].idxmin()
best_margin = region_metrics.loc[best_region, "Profit_Margin"]
worst_margin = region_metrics.loc[worst_region, "Profit_Margin"]
avg_margin = region_metrics["Profit_Margin"].mean().round(2)
worst_category = category_metrics["Profit_Margin"].idxmin()

# Build knowledge base chunks
knowledge_base = []

# Regional chunks
for region, row in region_metrics.iterrows():
    risks = risk_counts.get(region, 0)
    opps = opportunity_counts.get(region, 0)

    if row["Profit_Margin"] == best_margin:
        performance = (f"the best performing region with a margin "
                      f"of {row['Profit_Margin']}%")
    elif row["Profit_Margin"] == worst_margin:
        performance = (f"the weakest performing region with a margin "
                      f"of {row['Profit_Margin']}%, below the business "
                      f"average of {avg_margin}%")
    else:
        performance = (f"an average performing region with a margin "
                      f"of {row['Profit_Margin']}%")

    chunk = (
        f"{region} is {performance}. "
        f"Total Sales: {row['Total_Sales']}. "
        f"Total Profit: {row['Total_Profit']}. "
        f"Risk anomalies: {risks}. "
        f"Opportunity anomalies: {opps}. "
        f"Compared to the business average margin of {avg_margin}%, "
        f"this region is "
        f"{'below' if row['Profit_Margin'] < avg_margin else 'above'} average."
    )
    knowledge_base.append({"topic": f"region_{region}", "text": chunk})

# Category chunks
for category, row in category_metrics.iterrows():
    if row["Profit_Margin"] == category_metrics["Profit_Margin"].max():
        cat_label = "the highest margin category"
    elif row["Profit_Margin"] == category_metrics["Profit_Margin"].min():
        cat_label = "the lowest margin category and a priority concern"
    else:
        cat_label = "a mid-range margin category"

    chunk = (
        f"{category} is {cat_label} with a profit margin "
        f"of {row['Profit_Margin']}%. "
        f"Total Sales: {row['Total_Sales']}. "
        f"Total Profit: {row['Total_Profit']}. "
        f"Best category margin: {category_metrics['Profit_Margin'].max()}%. "
        f"Worst category margin: {category_metrics['Profit_Margin'].min()}%."
    )
    knowledge_base.append({"topic": f"category_{category}", "text": chunk})

# Overall summary chunk
total_sales = df_store["Sales"].sum().round(2)
total_profit = df_store["Profit"].sum().round(2)
overall_margin = (total_profit / total_sales * 100).round(2)
total_risk = len(df_store[df_store["Anomaly_Type"] == "Risk"])
total_opportunity = len(df_store[df_store["Anomaly_Type"] == "Opportunity"])

knowledge_base.append({
    "topic": "overall_summary",
    "text": (
        f"Overall business performance from 2014 to 2017. "
        f"Total transactions: {len(df_store)}. "
        f"Total sales: {total_sales}. "
        f"Total profit: {total_profit}. "
        f"Overall profit margin: {overall_margin}%. "
        f"Total risk anomalies: {total_risk}. "
        f"Total opportunity anomalies: {total_opportunity}. "
        f"Best performing region: {best_region} at {best_margin}% margin. "
        f"Worst performing region: {worst_region} at {worst_margin}% margin. "
        f"Most concerning category: {worst_category}."
    )
})

# Worst transaction chunk
worst = df_store[df_store["Anomaly_Type"] == "Risk"].nsmallest(
    1, "Anomaly_Score").iloc[0]
knowledge_base.append({
    "topic": "worst_transaction",
    "text": (
        f"Most anomalous risk transaction detected on "
        f"{worst['Order Date'].date()} in {worst['Region']} region, "
        f"{worst['Category']} category. "
        f"Sales: {round(worst['Sales'], 2)}. "
        f"Discount: {worst['Discount']}. "
        f"Profit: {round(worst['Profit'], 2)}. "
        f"Anomaly score: {round(worst['Anomaly_Score'], 4)}."
    )
})

# Save knowledge base as JSON
with open("../api/knowledge_base/chunks.json", "w") as f:
    json.dump(knowledge_base, f, indent=2)

print(f"Knowledge base saved with {len(knowledge_base)} chunks.")

Knowledge base saved with 9 chunks.


In [6]:
# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings from saved chunks
texts = [chunk["text"] for chunk in knowledge_base]
embeddings = embedding_model.encode(texts)

# Save embeddings as numpy array
# npy format is the standard way to save numpy arrays to disk
np.save("../api/knowledge_base/embeddings.npy", embeddings)

print(f"Embeddings saved. Shape: {embeddings.shape}")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2906.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings saved. Shape: (9, 384)


In [7]:
import os

# Check all required files exist
required_files = [
    "../api/models/fraud_model.pkl",
    "../api/models/scaler_fraud.pkl",
    "../api/models/anomaly_model.pkl",
    "../api/models/scaler_anomaly.pkl",
    "../api/knowledge_base/chunks.json",
    "../api/knowledge_base/embeddings.npy"
]

print("Checking saved files:\n")
all_good = True
for filepath in required_files:
    exists = os.path.exists(filepath)
    size = os.path.getsize(filepath) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"{status} -- {filepath} ({size:,} bytes)")
    if not exists:
        all_good = False

print()
if all_good:
    print("All files present. API is ready to be built.")
else:
    print("Some files are missing. Re-run the relevant cells above.")

Checking saved files:

OK -- ../api/models/fraud_model.pkl (1,208 bytes)
OK -- ../api/models/scaler_fraud.pkl (567 bytes)
OK -- ../api/models/anomaly_model.pkl (958,490 bytes)
OK -- ../api/models/scaler_anomaly.pkl (638 bytes)
OK -- ../api/knowledge_base/chunks.json (2,610 bytes)
OK -- ../api/knowledge_base/embeddings.npy (13,952 bytes)

All files present. API is ready to be built.
